# Create list of URLs and find manifests

Given the printout of the BibTeX reference for a collection, create a dataframe of entries and their IIIF manifests for use with various tools.

N.B. ensure webscraping has appropriate and considerate limits built in.

In [1]:
import json
import os
from pathlib import Path
import pandas as pd

In [8]:
import subprocess
import sys

import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin
import time


In [2]:
import bibtexparser
import pandas as pd

In [4]:
# Load and parse the BibTeX file
parser = bibtexparser.bparser.BibTexParser()
with open('1894.bib', 'r', encoding='utf-8') as bibfile:
    bibdata = bibtexparser.loads(bibfile.read(), parser=parser)

In [6]:
df = pd.DataFrame(bibdata.entries)
df.head()

,url,publisher,year,title,ENTRYTYPE,ID,abstract
0,https://digital.library.leeds.ac.uk/15890/,University of Leeds,1894,"Beverley, St. Mary's Church",misc,digitallibrary15890,NaN
1,https://digital.library.leeds.ac.uk/15885/,University of Leeds,1894,"Beverley, The Minster from E",misc,digitallibrary15885,NaN
2,https://digital.library.leeds.ac.uk/15884/,University of Leeds,1894,"Beverley, The Minster from S.E.",misc,digitallibrary15884,NaN
3,https://digital.library.leeds.ac.uk/15886/,University of Leeds,1894,"Beverley, The Minster from S.W.",misc,digitallibrary15886,NaN
4,https://digital.library.leeds.ac.uk/15887/,University of Leeds,1894,"Beverley, The Minster, Lady Chapel",misc,digitallibrary15887,NaN


In [10]:
def fetch_iiif_manifest(digital_library_url):
    """
    Fetch IIIF manifest URL from a Leeds digital library item page.
    
    Process:
    1. Fetch the digital library item page
    2. Find the link to "View more information about this item" (special-collections-explore)
    3. Fetch that page and extract the IIIF manifest URL
    
    Args:
        digital_library_url: URL like https://digital.library.leeds.ac.uk/15933/
    
    Returns:
        Manifest URL or None if not found
    """

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Connection': 'keep-alive',
    }
    
    try:
        # Step 1: Fetch initial digital library page
        response = requests.get(digital_library_url, timeout=10, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Step 2: Find the "View more information about this item" link
        more_info_link = None
        for link in soup.find_all('a', href=True):
            if 'special-collections-explore' in link['href']:
                more_info_link = link['href']
                break
        
        if not more_info_link:
            print(f"Could not find special-collections-explore link")
            return None
        
        # Ensure absolute URL
        if not more_info_link.startswith('http'):
            more_info_link = urljoin(digital_library_url, more_info_link)
        
        # Step 3: Fetch the special collections page with referer and additional headers
        headers_with_referer = headers.copy()
        headers_with_referer['Referer'] = digital_library_url
        
        response = requests.get(more_info_link, timeout=10, headers=headers_with_referer)
        
        # Don't raise on 403
        if response.status_code == 403:
            print(f"403 Forbidden")
        elif response.status_code != 200:
            response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Step 4: Extract the IIIF manifest URL
        # Look for the manifest link (usually contains iiif.library.leeds.ac.uk)
        for link in soup.find_all('a', href=True):
            href = link['href']
            if 'iiif.library.leeds.ac.uk' in href and 'presentation' in href:
                return href
        
        # Fallback: look in text for manifest URL pattern
        page_text = soup.get_text()
        match = re.search(r'https://iiif\.library\.leeds\.ac\.uk/presentation/[^\s\)\"\'<]+', page_text)
        if match:
            return match.group(0)
        
        return None
    except requests.exceptions.RequestException as e:
        print(f"Request Error: {e}")
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None

In [11]:
import time

def add_manifests_to_dataframe(df, url_column='related_artefacts', delay=1):
    """
    Add manifest URLs to dataframe by fetching from digital library pages.
    Includes delay to avoid overwhelming the server.
    
    Args:
        df: DataFrame with digital library URLs
        url_column: Column name containing the URLs
        delay: Seconds to wait between requests (default: 1)
    
    Returns:
        DataFrame with new 'manifest' column
    """
    df = df.copy()
    manifests = []
    
    for idx, url in enumerate(df[url_column]):
        print(f"[{idx+1}/{len(df)}] Fetching {url}...", end=" ")
        manifest = fetch_iiif_manifest(url)
        manifests.append(manifest)
        print(f"yes {manifest if manifest else 'no'}")
        time.sleep(delay)
    
    df['manifest'] = manifests
    return df



In [ ]:
df_with_manifests = add_manifests_to_dataframe(df, url_column="url", delay=15)

[1/204] Fetching https://digital.library.leeds.ac.uk/15890/... Request Error: HTTPSConnectionPool(host='explore.library.leeds.ac.uk', port=443): Read timed out. (read timeout=10)
yes no
[2/204] Fetching https://digital.library.leeds.ac.uk/15885/... Request Error: HTTPSConnectionPool(host='explore.library.leeds.ac.uk', port=443): Read timed out. (read timeout=10)
yes no
[3/204] Fetching https://digital.library.leeds.ac.uk/15884/... 

KeyboardInterrupt: 

In [ ]:
df_with_manifests